# Make your own adversarial images with L-BFGS-B

Open this notebook in **Google Colab**, select **Runtime → Change runtime type → GPU** if available, then run cells in order. CPU also works but can be slow. Internet is needed for dependencies, ImageNet weights, labels and the default image.

Based on Szegedy et al., [Intriguing properties of neural networks](https://arxiv.org/abs/1312.6199), §4.1. The attack targets a class you choose while penalizing the L2 distance from the input. This is a modern **ResNet-50 adaptation**, not a reproduction of the paper's AlexNet experiments or FGSM.

The paper approximates a minimum-distance targeted attack using box-constrained L-BFGS and a search over a penalty weight. Here we optimize

$$\min_{z\in[0,1]^m} c\sqrt{\|z-x\|_2^2+10^{-12}} + \mathrm{CE}(f(z),t).$$

We use a finite logarithmic sweep of `c`, rather than the paper's coefficient line search. We retain successful evaluated candidates; neither global minimality nor attack success is guaranteed. The small smoothing constant makes the norm differentiable at zero. Image pixels are RGB floats in [0,1]; model preprocessing remains inside the gradient computation.

**What you get:** original and adversarial PNGs at 224×224, a magnified perturbation plot, top-five predictions, distortion metrics, optimizer diagnostics, and an export ZIP. Success is checked again after PNG rounding. A changed prediction is an error only if the original semantic label is correct—inspect it yourself.

In [ ]:
%pip -q install "tensorflow>=2.16,<2.22" "scipy>=1.11,<2" "numpy>=1.26,<3" "pillow>=10,<13" "matplotlib>=3.8,<4" requests

In [ ]:
import io, json, platform, zipfile
from pathlib import Path
import numpy as np
import tensorflow as tf
import scipy
from scipy.optimize import minimize, Bounds
from PIL import Image, ImageOps
import matplotlib.pyplot as plt
import requests
from IPython.display import display
from tensorflow.keras.applications.resnet50 import ResNet50, preprocess_input, decode_predictions

tf.keras.utils.set_random_seed(7)
print("Python", platform.python_version(), "TensorFlow", tf.__version__, "SciPy", scipy.__version__)
print("GPU devices:", tf.config.list_physical_devices('GPU'))

## 1. Choose your image
Use a direct image URL, a local file path, or set `SOURCE = "upload"` to upload in Colab. The image is EXIF-corrected, converted to RGB, resized to 224×224, and rounded to an 8-bit baseline **before** attacking. Distortion is measured against this baseline, not the full-resolution source. Transparent images are composited onto white.

In [ ]:
SOURCE = "https://raw.githubusercontent.com/jrballesteros/adversarial_attacks/main/cat.png"
# SOURCE = "upload"          # Colab file picker
# SOURCE = "/content/my_photo.png"

if SOURCE == "upload":
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Upload exactly one image.")
    raw = io.BytesIO(next(iter(uploaded.values())))
elif SOURCE.startswith(("https://", "http://")):
    response = requests.get(SOURCE, timeout=60)
    response.raise_for_status()
    raw = io.BytesIO(response.content)
else:
    raw = SOURCE
with Image.open(raw) as im:
    im = ImageOps.exif_transpose(im).convert("RGBA")
    background = Image.new("RGBA", im.size, "white")
    background.alpha_composite(im)
    im = background.convert("RGB").resize((224, 224), Image.Resampling.BILINEAR)
    x = np.asarray(im, dtype=np.float32) / 255.0
display(im)
assert x.shape == (224, 224, 3) and np.isfinite(x).all()

## 2. Load ResNet-50 and choose a target
The model uses ImageNet's 1,000 classes. `TARGET` can be an exact label (for example `"ostrich"`) or an integer from 0 to 999. The default mirrors the ostrich target in the paper's Figure 5. The source model must initially predict a different class.

In [ ]:
model = ResNet50(weights="imagenet", classifier_activation=None)
model.trainable = False
# Decode one-hot rows to obtain Keras' exact index-to-label mapping.
labels = [row[0][1] for row in decode_predictions(np.eye(1000, dtype=np.float32), top=1)]

@tf.function(input_signature=[tf.TensorSpec([None,224,224,3], tf.float32)])
def logits_for(images):
    return model(preprocess_input(images * 255.0), training=False)

def probabilities(image):
    return tf.nn.softmax(logits_for(tf.convert_to_tensor(image[None], tf.float32)), axis=-1).numpy()[0]

def show_top5(image):
    p = probabilities(image)
    for i in np.argsort(p)[-5:][::-1]:
        print(f"{i:3d}  {labels[i]:25s} {p[i]:.2%}")
    return p

print("Original predictions:")
p_original = show_top5(x)
original_class = int(p_original.argmax())

TARGET = "ostrich"  # Change this, then rerun this cell and everything below.
if isinstance(TARGET, str):
    if TARGET not in labels:
        raise ValueError("Unknown exact label. Search the labels list for your desired class.")
    target = labels.index(TARGET)
else:
    target = int(TARGET)
assert 0 <= target < 1000
if target == original_class:
    raise ValueError("Choose a target different from the original top prediction.")
print("Target:", target, labels[target])
# Find other labels, e.g.: [ (i,s) for i,s in enumerate(labels) if 'dog' in s ]

## 3. Optimize the image
`c` controls the penalty for changing pixels: larger values usually favor smaller changes. Each run starts from the original image. The default runs five solves with up to 100 iterations each; this can take several minutes or longer on CPU. Increase `MAXITER` or extend `C_VALUES` if no candidate succeeds. Inspect optimizer messages: reaching an iteration limit is distinct from whether the target prediction was achieved.

The code tracks target-successful images at all objective evaluations, including line-search evaluations. Selection prioritizes candidates that remain successful after 8-bit rounding, then chooses the smallest L2 distance among those observed. This is a practical finite search, not proof of the minimum perturbation.

In [ ]:
C_VALUES = [10.0, 1.0, 0.1, 0.01, 0.001]
MAXITER = 100
x_tf = tf.constant(x)
best_continuous = None
best_png = None
best_fallback = None
history = []

def rounded(a):
    return np.rint(np.clip(a, 0, 1) * 255).astype(np.uint8).astype(np.float32) / 255.0

def l2(a):
    return float(np.linalg.norm((a - x).astype(np.float64)))

@tf.function(input_signature=[tf.TensorSpec([224,224,3], tf.float32), tf.TensorSpec([], tf.float32)])
def value_and_grad(z, c):
    with tf.GradientTape() as tape:
        tape.watch(z)
        logits = logits_for(z[None])
        ce = tf.nn.sparse_softmax_cross_entropy_with_logits(labels=[target], logits=logits)[0]
        distance = tf.sqrt(tf.reduce_sum(tf.square(z - x_tf)) + 1e-12)
        objective = c * distance + ce
    return objective, tape.gradient(objective, z), logits[0]

for c_value in C_VALUES:
    seen = {"continuous": None, "fallback": None}
    def objective(flat):
        z = flat.reshape(x.shape).astype(np.float32)
        value, grad, logits = value_and_grad(tf.constant(z), tf.constant(c_value, tf.float32))
        lv = logits.numpy()
        margin = float(lv[target] - np.max(np.delete(lv, target)))
        if seen["fallback"] is None or margin > seen["fallback"][0]:
            seen["fallback"] = (margin, z.copy())
        if int(lv.argmax()) == target:
            distance = l2(z)
            if seen["continuous"] is None or distance < seen["continuous"][0]:
                seen["continuous"] = (distance, z.copy())
        if not np.isfinite(value.numpy()) or not np.isfinite(grad.numpy()).all():
            raise FloatingPointError("Non-finite objective or gradient.")
        return float(value.numpy()), grad.numpy().astype(np.float64).ravel()

    result = minimize(objective, x.astype(np.float64).ravel(), jac=True,
                      method="L-BFGS-B", bounds=Bounds(0.0, 1.0),
                      options={"maxiter": MAXITER, "maxfun": MAXITER*12,
                               "maxls": 30, "maxcor": 10, "ftol": 1e-10, "gtol": 1e-6})
    candidate = np.clip(result.x.reshape(x.shape), 0, 1).astype(np.float32)
    # Check final iterate plus the best observed continuous success.
    candidates = [candidate]
    if seen["continuous"] is not None:
        candidates.append(seen["continuous"][1])
    for a in candidates:
        if int(probabilities(a).argmax()) == target:
            if best_continuous is None or l2(a) < l2(best_continuous):
                best_continuous = a.copy()
        q = rounded(a)
        if int(probabilities(q).argmax()) == target:
            if best_png is None or l2(q) < l2(best_png):
                best_png = q.copy()
    if best_fallback is None or seen["fallback"][0] > best_fallback[0]:
        best_fallback = seen["fallback"]
    record = {"c": c_value, "iterations": int(result.nit), "evaluations": int(result.nfev),
              "optimizer_converged": bool(result.success), "message": str(result.message),
              "final_target_success": bool(probabilities(candidate).argmax() == target),
              "best_png_l2_so_far": None if best_png is None else l2(best_png)}
    history.append(record)
    print(record)

if best_png is not None:
    adversarial = best_png
    print("SUCCESS: a target-successful 8-bit candidate was found.")
elif best_continuous is not None:
    adversarial = rounded(best_continuous)
    print("Continuous attack succeeded, but PNG rounding removed the target prediction.")
else:
    adversarial = rounded(best_fallback[1])
    print("No targeted success found. Displaying the highest target-margin candidate; do not label it successful.")

## 4. Compare and export
PNG is lossless, but converting floats to 8-bit pixels can still remove an attack. We save and reload the actual file, then report its prediction and distortion. The perturbation panel is magnified and shifted around gray; it is not the image to upload.

In [ ]:
output = Path("adversarial_outputs")
output.mkdir(exist_ok=True)
Image.fromarray(np.rint(x * 255).astype(np.uint8)).save(output / "original.png")
Image.fromarray(np.rint(adversarial * 255).astype(np.uint8)).save(output / "adversarial.png")
with Image.open(output / "adversarial.png") as saved:
    reloaded = np.asarray(saved.convert("RGB"), dtype=np.float32) / 255.0
p_saved = probabilities(reloaded)
delta = reloaded - x
metrics = {
    "target": labels[target], "target_index": target,
    "original_prediction": labels[original_class],
    "saved_prediction": labels[int(p_saved.argmax())],
    "targeted_success_after_reload": bool(p_saved.argmax() == target),
    "prediction_changed_after_reload": bool(p_saved.argmax() != original_class),
    "target_probability": float(p_saved[target]),
    "l2": l2(reloaded), "rms": float(np.sqrt(np.mean(delta**2))),
    "linf": float(np.max(np.abs(delta))),
    "pixel_scale": "RGB [0,1]", "source": SOURCE,
    "tensorflow": tf.__version__, "scipy": scipy.__version__,
    "c_values": C_VALUES, "maxiter": MAXITER, "optimizer_runs": history,
}
print(json.dumps(metrics, indent=2))
print("Saved PNG predictions:")
show_top5(reloaded)
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, image, title in zip(axes, [x, np.clip(0.5 + 10*delta, 0, 1), reloaded],
    [f"Original: {labels[original_class]}", "Perturbation ×10 + gray", f"Saved PNG: {labels[int(p_saved.argmax())]}"]):
    ax.imshow(image); ax.set_title(title); ax.axis("off")
plt.tight_layout()
fig.savefig(output / "comparison.png", dpi=160, bbox_inches="tight")
plt.show()
np.save(output / "perturbation.npy", delta)
(output / "metrics.json").write_text(json.dumps(metrics, indent=2))
archive = Path("adversarial_images.zip")
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as bundle:
    for file in sorted(output.iterdir()):
        bundle.write(file, arcname=file.name)
print("Export:", archive.resolve())
try:
    from google.colab import files
    files.download(str(archive))
except ImportError:
    display(__import__('IPython').display.FileLink(str(archive)))

## 5. Try it in the Keras.js portal
1. Download the ZIP and extract `adversarial.png` and `original.png`.
2. Upload both PNGs to your GitHub repository without resizing or recompressing them.
3. Open each file's **Raw** link. A direct URL has this form: `https://raw.githubusercontent.com/USER/REPOSITORY/main/adversarial.png`.
4. In [the ResNet-50 demo](https://transcranial.github.io/keras-js/#/resnet50), wait until the weights load, paste the raw URL and press Return. Test the baseline first and compare its top predictions with this notebook, then test the adversarial PNG.

The notebook attacks the loaded Keras model. The older Keras.js demo may use different weights or preprocessing, so transfer is **an experiment**, not a guarantee. Even a successful local attack can fail in the demo. PNG helps preserve pixels; avoid JPEG, screenshots, image editors, or hosts that transform the file.

**Troubleshooting:** if the target is not reached, try `MAXITER = 250`, add intermediate/smaller `c` values, or choose another target. Larger distortions can become visible; judge the image as well as RMS/L2. The selected result is the smallest successful candidate observed, not necessarily imperceptible. To process another image, change `SOURCE` and rerun from section 1 onward. Restart or rerun sections 2–4 after changing `TARGET`.

## References
- Szegedy et al. (2014), [Intriguing properties of neural networks](https://arxiv.org/abs/1312.6199), §4.1 and Figure 5.
- [Keras ResNet-50 model and preprocessing](https://keras.io/api/applications/resnet/resnet_models/).
- [SciPy L-BFGS-B](https://docs.scipy.org/doc/scipy/reference/optimize.minimize-lbfgsb.html).

Validation at creation: notebook format and Python cell syntax checked. The full pretrained-model experiment was not executed in the authoring environment; TensorFlow and model weights were unavailable there. Run all cells in Colab to obtain and verify your own results.